In [101]:
!pip install numpy pandas qiskit-aer qiskit-algorithms qiskit-machine-learning pandas pylatexenc ucimlrepo qiskit-ibm-runtime qiskit-machine-learning qiskit-aer


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [102]:
!pip install xgboost catboost


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [103]:
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
import warnings
from qiskit_machine_learning.kernels import FidelityStatevectorKernel
from qiskit_algorithms.state_fidelities import ComputeUncompute
from qiskit_machine_learning.algorithms import QSVC
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
from qiskit.transpiler import generate_preset_pass_manager
warnings.filterwarnings('ignore')

In [ ]:
# from qiskit_ibm_runtime import QiskitRuntimeService

# QiskitRuntimeService.save_account(
#   token=token,
#   overwrite=True
# )

In [105]:
import itertools
class QuantumKernelCircuits:

    lambda_=1.0
    
    @staticmethod
    def create_iqp_full(n_qubits, params):
        # params = ParameterVector("x", n_qubits)
        qc = QuantumCircuit(n_qubits)
        
        # Hadamard layer
        for i in range(n_qubits):
            qc.h(i)
            
        # Single-qubit rotation layer
        for i in range(n_qubits):
            qc.rz(params[i], i)
            
        # Full entanglement layer
        for i, j in itertools.combinations(range(n_qubits), 2):
            theta = QuantumKernelCircuits.lambda_ * params[i] * params[j]
            qc.rzz(theta, i, j)
                
        return qc
    
    @staticmethod
    def create_iqp_linear(n_qubits, params):
        qc = QuantumCircuit(n_qubits)
        
        # Hadamard layer
        for i in range(n_qubits):
            qc.h(i)
            
        # Single-qubit rotation layer
        for i in range(n_qubits):
            qc.rz(params[i], i)
            
        # Linear entanglement layer
        for i in range(n_qubits - 1):
            theta = QuantumKernelCircuits.lambda_ * params[i] * params[i+1]
            qc.rzz(theta, i, i+1)
        #     qc.cx(i, i+1)
        #     qc.rz(x[i] * x[i+1], i+1)
        #     qc.cx(i, i+1)
                
        return qc
    
    @staticmethod
    def create_iqp_circular(n_qubits, params):
        qc = QuantumCircuit(n_qubits)
        
        # Hadamard layer
        for i in range(n_qubits):
            qc.h(i)
            
        # Single-qubit rotation layer
        for i in range(n_qubits):
            qc.rz(params[i], i)
            
        # Circular entanglement layer
        for i in range(n_qubits):
            j = (i + 1) % n_qubits
            theta = QuantumKernelCircuits.lambda_ * params[i] * params[j]
            qc.rzz(theta, i, j)
                
        return qc
    
    @staticmethod
    def create_pauli_x(n_qubits, params):
        qc = QuantumCircuit(n_qubits)
        
        for i in range(n_qubits):
            qc.rx(params[i], i)
            
        return qc
    
    @staticmethod
    def create_pauli_y(n_qubits, params):
        qc = QuantumCircuit(n_qubits)
        
        for i in range(n_qubits):
            qc.ry(params[i], i)
            
        return qc
    
    @staticmethod
    def create_pauli_z(n_qubits, params):
        qc = QuantumCircuit(n_qubits)
        
        for i in range(n_qubits):
            qc.h(i)
            qc.rz(params[i], i)
            
        return qc

In [106]:
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.algorithms import QSVC
from qiskit import transpile
from qiskit_aer import AerSimulator

class QuantumKernelEstimator:
    
    def __init__(self, kernel_type='full', n_measurements=1024):
        self.kernel_type = kernel_type
        self.n_measurements = n_measurements
        self.backend = None
        
        # Map kernel types to circuit creation functions
        self.circuit_creators = {
            'full': QuantumKernelCircuits.create_iqp_full,
            'linear': QuantumKernelCircuits.create_iqp_linear,
            'circular': QuantumKernelCircuits.create_iqp_circular,
            'pauli_x': QuantumKernelCircuits.create_pauli_x,
            'pauli_y': QuantumKernelCircuits.create_pauli_y,
            'pauli_z': QuantumKernelCircuits.create_pauli_z
        }

    def _build_feature_map(self, n_features):
        params = ParameterVector('x', length=n_features)
        creator_fn = self.circuit_creators[self.kernel_type]
        feature_map = creator_fn(n_features, params)
        
        return feature_map
    
    def build_quantum_kernel(self, n_features, use_hardware=False):
        self._feature_map = self._build_feature_map(n_features)

        # create FQK
        if use_hardware:
            # aer_sim = AerSimulator()
            service = QiskitRuntimeService()
            self.backend = service.least_busy(simulator=False, operational=True)
            
            print("Using backend:", self.backend.name)

            pm = generate_preset_pass_manager(backend=self.backend)
            isa_feature_map = pm.run(self._feature_map)

            self.sampler = SamplerV2(mode=self.backend)
            self.sampler.options.default_shots = int(self.n_measurements)
            
            # create fidelity implementation
            fidelity = ComputeUncompute(sampler=self.sampler, transpiler=pm)
            qkernel = FidelityQuantumKernel(
                feature_map=isa_feature_map, 
                fidelity=fidelity, 
                max_circuits_per_job=300
            )
        else:
            transpiled_feature_map = transpile(self._feature_map, AerSimulator())
            qkernel = FidelityStatevectorKernel(feature_map=transpiled_feature_map)

        return qkernel

In [107]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)
from ucimlrepo import fetch_ucirepo
import time

In [ ]:
def load_dataset(feature_cols, target_cols, manual=False, data='500', path=None):

    if manual:
        if data == '500':
            train_data = pd.read_csv('./dataset/dataset-500-manual/dataset_500_manual_train.csv')
            val_data = pd.read_csv('./dataset/dataset-500-manual/dataset_500_manual_val.csv')
            test_data = pd.read_csv('./dataset/dataset-500-manual/dataset_500_manual_test.csv')
        elif data == '1000':
            train_data = pd.read_csv('./dataset/dataset-1000-manual/dataset_1000_manual_train.csv')
            val_data = pd.read_csv('./dataset/dataset-1000-manual/dataset_1000_manual_val.csv')
            test_data = pd.read_csv('./dataset/dataset-1000-manual/dataset_1000_manual_test.csv')
        elif data == '5000':
            train_data = pd.read_csv('./dataset/dataset-5000-manual/dataset_5000_manual_train.csv')
            val_data = pd.read_csv('./dataset/dataset-5000-manual/dataset_5000_manual_val.csv')
            test_data = pd.read_csv('./dataset/dataset-5000-manual/dataset_5000_manual_test.csv')
        elif data == '10000':
            train_data = pd.read_csv('./dataset/dataset-10000-manual/dataset_10000_manual_train.csv')
            val_data = pd.read_csv('./dataset/dataset-10000-manual/dataset_10000_manual_val.csv')
            test_data = pd.read_csv('./dataset/dataset-10000-manual/dataset_10000_manual_test.csv')
        else:
            raise ValueError("Invalid data size. Choose from '500', '1000', '5000', or '10000'.")
        
        X_train = train_data[feature_cols]
        y_train = train_data[target_cols]

        X_val = val_data[feature_cols]
        y_val = val_data[target_cols]

        X_test = test_data[feature_cols]
        y_test = test_data[target_cols]
    else:
        df = pd.read_csv(path)
        
        X = df[feature_cols]
        y = df[target_cols]

        # random_state = [42, x, x, x,]
        # train:temp (70%:30%)
        X_train, X_temp, y_train, y_temp = train_test_split(
            X, y, test_size=0.3, stratify=y, random_state=42
        )

        # temp -> val:test = 15:15
        X_val, X_test, y_val, y_test = train_test_split(
            X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
        )
    
    return X_train, X_val, X_test, y_train, y_val, y_test

In [109]:
def preprocess(data):
    from sklearn.preprocessing import LabelEncoder

    # reset index
    X_train = data['X_train'].reset_index(drop=True)
    X_val  = data['X_val'].reset_index(drop=True)
    X_test  = data['X_test'].reset_index(drop=True)
    y_train = data['y_train'].reset_index(drop=True)
    y_val = data['y_val'].reset_index(drop=True)
    y_test  = data['y_test'].reset_index(drop=True)

    cat_features = [
        i for i, col in enumerate(X_train.columns)
        if X_train[col].dtype == 'object'
    ]

    label_encoder = LabelEncoder()
    y_train = label_encoder.fit_transform(y_train)
    y_val = label_encoder.fit_transform(y_val)
    y_test = label_encoder.fit_transform(y_test)

    # z score normalization
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)
    X_test = scaler.transform(X_test)

    data_processed = {
        'X_train': X_train,
        'X_val': X_val,
        'X_test': X_test,
        'y_train': y_train,
        'y_val': y_val,
        'y_test': y_test,
        'cat_features': cat_features,
    }

    return data_processed

In [110]:
def train_model(data, kernels, use_cv=False, model_type='quantum', use_hardware=False, verbose=True):
    from sklearn.svm import SVC
    results = []
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    # Cross Validation
    for kernel in kernels:
        if verbose:
            print(f"\n{'='*60}")
            print(f"Evaluating kernel: {kernel}")
            print(f"{'='*60}")

        start_time = time.time()
        if use_cv:
            # Menyimpan hasil dari setiap fold
            fold_results = []
            fold_num = 1
        
            # ITERASI SETIAP FOLD
            for train_idx, val_idx in cv.split(data['X_train'], data['y_train']):
                print(f"\n--- Fold {fold_num}/5 ---")
                
                # Split data untuk fold ini
                X_fold_train = data['X_train'][train_idx]
                y_fold_train = data['y_train'][train_idx]
                X_fold_val = data['X_train'][val_idx]
                y_fold_val = data['y_train'][val_idx]
                
                # Train model untuk fold ini
                model = QSVC(kernel_type=kernel)
                
                # TRAIN
                train_start = time.time()
                model.fit(X_fold_train, y_fold_train)
                train_time = time.time() - train_start
                
                # Validasi pada fold ini
                val_start = time.time()
                pred_val = model.predict(X_fold_val)
                val_acc = accuracy_score(y_fold_val, pred_val)
                val_time = time.time() - val_start

                total_time = time.time() - start_time
                total_jobs = model.kernel_estimator.get_job_count()

                fold_results.append({
                    'train_time': train_time,
                    'val_time': val_time,
                    'val_accuracy': val_acc
                })
                
                print(f"Fold {fold_num} - Val Accuracy: {val_acc:.4f}")
                fold_num += 1

            # Hitung rata-rata dari semua fold
            avg_train_time = np.mean([f['train_time'] for f in fold_results])
            avg_val_time = np.mean([f['val_time'] for f in fold_results])
            avg_val_acc = np.mean([f['val_accuracy'] for f in fold_results])
            std_val_acc = np.std([f['val_accuracy'] for f in fold_results])

        else:
            if model_type == 'quantum':
                kernel_instance = QuantumKernelEstimator(kernel_type=kernel, n_measurements=512)
                feature_map = kernel_instance.build_quantum_kernel(n_features=data['X_train'].shape[1], use_hardware=use_hardware)
                model = QSVC(quantum_kernel=feature_map, C=1.0, decision_function_shape='ovr')

            elif model_type == 'sklearn':
                model = SVC(kernel=kernel, C=1.0)

            elif model_type == 'xgboost':
                from xgboost import XGBClassifier
                model = XGBClassifier(n_estimators=500, max_depth=10, subsample=0.8, learning_rate=1, objective='binarry:logistic', eval_metric='mlogloss', random_state=42)

            elif model_type == 'catboost':
                from catboost import CatBoostClassifier
                cat_features = data['cat_features']
                model = CatBoostClassifier(
                    loss_function="MultiClass",
                    iterations=1000,
                    learning_rate=0.1,
                    depth=6,
                    eval_metric="Accuracy",
                    verbose=100,
                    random_seed=42,
                )

            # TRAIN
            train_start = time.time()
            if verbose:
                print("Training start.")
            
            if model_type == 'xgboost':
                model.fit(
                    data['X_train'], 
                    data['y_train'], 
                    eval_set=[(data['X_val'], data['y_val'])], 
                    verbose=False
                )
            elif model_type == 'catboost':
                model.fit(
                    data['X_train'],
                    data['y_train'],
                    cat_features=cat_features,
                    eval_set=(data['X_val'], data['y_val']),
                    early_stopping_rounds=50
                )
            else:
                model.fit(data['X_train'], data['y_train'])
            
            if verbose:
                print("Training completed.")
                
            train_time = time.time() - train_start

            # VAL
            val_start = time.time()
            if verbose:
                print("Val start.")
            y_val_pred = model.predict(data['X_val'])
            if verbose:
                print("Val completed.")
            val_time = time.time() - val_start
            val_acc = accuracy_score(data['y_val'], y_val_pred)

            # TEST
            test_start = time.time()
            if verbose:
                print("Test start.")
            y_test_pred = model.predict(data['X_test'])
            if verbose:
                print("Test completed.")
            test_time = time.time() - test_start
            
        total_time = train_time + val_time + test_time

        report = classification_report(data['y_test'], y_test_pred, output_dict=True)
        results.append({
            'kernel': kernel,
            'val_accuracy': val_acc,
            'accuracy': report['accuracy'],
            'precision': report['weighted avg']['precision'],
            'recall': report['weighted avg']['recall'],
            'f1-score': report['weighted avg']['f1-score'],
            'train_time': train_time,
            'val_time': val_time,
            'test_time': test_time,
            'total_time': total_time,
        })

    return results

In [111]:
def report(results, data_metadata):
    print(f"N_train = {len(data_metadata['X_train'])}")
    print(f"N_val   = {len(data_metadata['X_val'])}")
    print(f"N_test = {len(data_metadata['X_test'])}")

    print(f"\n{'='*110}")
    distribution_df = pd.DataFrame({
        'Train': data_metadata['y_train'].value_counts(),
        'Validation': data_metadata['y_val'].value_counts(),
        'Test': data_metadata['y_test'].value_counts()
    }).fillna(0).astype(int)

    print("\n=== KOMPOSISI KELAS DATASET ===")
    print(distribution_df)

    # --- TABEL RINGKASAN AKHIR ---
    print(f"\n{'='*150}")
    print(f"{'PERFORMANCE SUMMARY - QUANTUM SVC':^110}")
    print(f"{'='*150}")

    # Header
    header = f"{'Kernel':<12} | {'Val Acc':<8} | {'Acc':<8} | {'Prec':<8} | {'Recall':<8} | {'F1':<8} | {'Train(s)':<10} | {'Val(s)':<10} | {'Test(s)':<10} | {'Total(s)':<10}"
    print(header)
    print("-" * 150)

    for r in results:
        print(f"{r['kernel']:<12} | "
            f"{r['val_accuracy']:<8.3f} | "
            f"{r['accuracy']:<8.3f} | "
            f"{r['precision']:<8.3f} | "
            f"{r['recall']:<8.3f} | "
            f"{r['f1-score']:<8.3f} | "
            f"{r['train_time']:<10.2f} | "
            f"{r['val_time']:<10.2f} | "
            f"{r['test_time']:<10.2f} | "
            f"{r['total_time']:<10.2f}")

    print("=" * 110)

In [112]:
dataset_path = './dataset/Dataset_TehHijau.csv'
feature_cols = [
        "MQ3", 
        "TGS822", 
        "TGS2602", 
        "MQ5", 
        "MQ138", 
        "TGS2620", 
        "TGS813", 
        "TGS2600", 
        "TGS2611", 
        "TGS2603",
        "Humidity",
        "Celsius",
    ]
target_cols = "Kategori"

quantum_kernel_types = [
    'full', 
    'linear', 
    'circular', 
    'pauli_x', 
    'pauli_y', 
    'pauli_z'
]
kernel_types = [
    'linear', 
    'poly', 
    'rbf', 
    'sigmoid'
]

X_train, X_val, X_test, y_train, y_val, y_test = load_dataset(
    feature_cols, 
    target_cols, 
    # manual=True, 
    path=dataset_path
    # data='500'
)
data = {
    'X_train': X_train,
    'X_val': X_val,
    'X_test': X_test,
    'y_train': y_train,
    'y_val': y_val,
    'y_test': y_test
}

### SVC

In [113]:
processed_data = preprocess(data)
result = train_model(
    processed_data, 
    kernel_types, 
    use_cv=False, 
    model_type='sklearn', 
    # use_hardware=False, 
    verbose=False
)
report(result, data_metadata=data)

N_train = 7286
N_val   = 1561
N_test = 1562


=== KOMPOSISI KELAS DATASET ===
          Train  Validation  Test
Kategori                         
E          3254         697   698
D          1622         347   348
C          1601         343   343
A           649         139   139
B           160          35    34

                                      PERFORMANCE SUMMARY - QUANTUM SVC                                       
Kernel       | Val Acc  | Acc      | Prec     | Recall   | F1       | Train(s)   | Val(s)     | Test(s)    | Total(s)  
------------------------------------------------------------------------------------------------------------------------------------------------------
linear       | 0.664    | 0.681    | 0.692    | 0.681    | 0.671    | 2.72       | 0.21       | 0.20       | 3.13      
poly         | 0.880    | 0.887    | 0.894    | 0.887    | 0.885    | 1.18       | 0.14       | 0.14       | 1.46      
rbf          | 0.985    | 0.974    | 0.974    | 0.974    | 0.

### XGBoost

In [114]:
processed_data = preprocess(data)
result = train_model(
    processed_data, 
    kernel_types, 
    use_cv=False, 
    model_type='xgboost', 
    # use_hardware=False, 
    verbose=False
)
report(result, data_metadata=data)

N_train = 7286
N_val   = 1561
N_test = 1562


=== KOMPOSISI KELAS DATASET ===
          Train  Validation  Test
Kategori                         
E          3254         697   698
D          1622         347   348
C          1601         343   343
A           649         139   139
B           160          35    34

                                      PERFORMANCE SUMMARY - QUANTUM SVC                                       
Kernel       | Val Acc  | Acc      | Prec     | Recall   | F1       | Train(s)   | Val(s)     | Test(s)    | Total(s)  
------------------------------------------------------------------------------------------------------------------------------------------------------
linear       | 0.999    | 0.998    | 0.998    | 0.998    | 0.998    | 1.92       | 0.01       | 0.01       | 1.93      
poly         | 0.999    | 0.998    | 0.998    | 0.998    | 0.998    | 2.38       | 0.01       | 0.01       | 2.40      
rbf          | 0.999    | 0.998    | 0.998    | 0.998    | 0.

In [115]:
processed_data = preprocess(data)
result = train_model(
    processed_data, 
    kernel_types, 
    use_cv=False, 
    model_type='catboost', 
    # use_hardware=False, 
    verbose=False
)
report(result, data_metadata=data)

0:	learn: 0.6438375	test: 0.6598334	best: 0.6598334 (0)	total: 7.3ms	remaining: 7.3s
100:	learn: 0.9954708	test: 0.9948751	best: 0.9948751 (99)	total: 657ms	remaining: 5.85s
200:	learn: 0.9994510	test: 0.9993594	best: 0.9993594 (166)	total: 1.36s	remaining: 5.39s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.999359385
bestIteration = 166

Shrink model to first 167 iterations.
0:	learn: 0.6438375	test: 0.6598334	best: 0.6598334 (0)	total: 7.24ms	remaining: 7.23s
100:	learn: 0.9954708	test: 0.9948751	best: 0.9948751 (99)	total: 642ms	remaining: 5.71s
200:	learn: 0.9994510	test: 0.9993594	best: 0.9993594 (166)	total: 1.32s	remaining: 5.25s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.999359385
bestIteration = 166

Shrink model to first 167 iterations.
0:	learn: 0.6438375	test: 0.6598334	best: 0.6598334 (0)	total: 6.38ms	remaining: 6.37s
100:	learn: 0.9954708	test: 0.9948751	best: 0.9948751 (99)	total: 638ms	remaining: 5.68s
200:	learn: 0.999451

In [ ]:
processed_data = preprocess(data)
result = train_model(
    processed_data, 
    kernels=quantum_kernel_types, 
    use_cv=False, 
    model_type='quantum', 
    # use_hardware=False, 
    verbose=True
)
report(result, data_metadata=data)